# Importations

In [1]:
# Numerical and scientific python programming
import numpy as np
import sympy as sp

from IPython.display import display

from moments.bloch import (compute_pauli_basis, compute_tensor_basis, compute_subset_index_map,
                           compute_bloch_vector, compute_dm_from_bloch, compute_bloch_norms_from_dm, compute_bloch_norms_from_vector,)

# Basis generation

In [2]:
basis = compute_pauli_basis()

print("Pauli basis shape:", basis.shape)
print("\n Basis elements:")
for idx, s in zip(["I", "X", "Y", "Z"], basis):
    print("\n" + str(idx))
    display(sp.Matrix(s))

Pauli basis shape: (4, 2, 2)

 Basis elements:

I


Matrix([
[1.0,   0],
[  0, 1.0]])


X


Matrix([
[  0, 1.0],
[1.0,   0]])


Y


Matrix([
[    0, -1.0*I],
[1.0*I,      0]])


Z


Matrix([
[1.0,    0],
[  0, -1.0]])

In [3]:
N = 3
d = 2**N

local_bases = [basis] * N
local_basis_sizes = [len(basis) for basis in local_bases]

tensor_basis = compute_tensor_basis(local_bases)
subset_index_map = compute_subset_index_map(local_basis_sizes)

idx = 1
print("Tensor basis shape:", tensor_basis.shape)
print("\n Basis element:", idx)
display(sp.Matrix(tensor_basis[idx-1]))

for subset, indices in subset_index_map.items():
    print(subset, "-->", indices)

Tensor basis shape: (64, 8, 8)

 Basis element: 1


Matrix([
[1.0,   0,   0,   0,   0,   0,   0,   0],
[  0, 1.0,   0,   0,   0,   0,   0,   0],
[  0,   0, 1.0,   0,   0,   0,   0,   0],
[  0,   0,   0, 1.0,   0,   0,   0,   0],
[  0,   0,   0,   0, 1.0,   0,   0,   0],
[  0,   0,   0,   0,   0, 1.0,   0,   0],
[  0,   0,   0,   0,   0,   0, 1.0,   0],
[  0,   0,   0,   0,   0,   0,   0, 1.0]])

(1,) --> [16 32 48]
(2,) --> [ 4  8 12]
(3,) --> [1 2 3]
(1, 2) --> [20 24 28 36 40 44 52 56 60]
(1, 3) --> [17 18 19 33 34 35 49 50 51]
(2, 3) --> [ 5  6  7  9 10 11 13 14 15]
(1, 2, 3) --> [21 22 23 25 26 27 29 30 31 37 38 39 41 42 43 45 46 47 53 54 55 57 58 59
 61 62 63]


# Bloch vector and density-matrix reconstruction

In [4]:
rng = np.random.default_rng()

r = {}
for subset, indices in subset_index_map.items():
    r[subset] = rng.normal(size=len(indices))

rho = compute_dm_from_bloch(tensor_basis, subset_index_map, r)

r_rec = compute_bloch_vector(tensor_basis, subset_index_map, rho)

coincide = []
difference = []


for subset in subset_index_map.keys():
    difference.append(float(np.linalg.norm(r[subset] - r_rec[subset])))
    coincide.append(np.allclose(r[subset], r_rec[subset]))

print("Bloch vectors coincide:", all(coincide))
print("Total difference:", sum(difference))

Bloch vectors coincide: True
Total difference: 2.544111447591873e-15


In [5]:
rng = np.random.default_rng()

X = rng.normal(size=(d, d)) + 1j * rng.normal(size=(d, d))
rho = X @ X.conj().T
tr_val = np.trace(rho).real
rho = rho/tr_val

r = compute_bloch_vector(tensor_basis, subset_index_map, rho)

rho_rec = compute_dm_from_bloch(tensor_basis, subset_index_map, r)

print("Density matrices coincide:", np.allclose(rho, rho_rec))
print("Total difference:", np.linalg.norm(rho - rho_rec))

Density matrices coincide: True
Total difference: 6.013703299064972e-17


# Bloch norms computation

In [6]:
rng = np.random.default_rng()

r = {}
R = {}
for subset, indices in subset_index_map.items():
    r_M = rng.normal(size=len(indices))
    r[subset] = r_M.copy()
    R[subset] = np.linalg.norm(r_M)

rho = compute_dm_from_bloch(tensor_basis, subset_index_map, r)
r_rec = compute_bloch_vector(tensor_basis, subset_index_map, rho)

R_rho = compute_bloch_norms_from_dm(tensor_basis, subset_index_map, rho)

coincide = []
difference = []

for subset in subset_index_map.keys():
    difference.append(float(np.linalg.norm(R[subset] - R_rho[subset])))
    coincide.append(np.allclose(R[subset], R_rho[subset]))

print("Bloch norms coincide:", all(coincide))
print("Total difference:", sum(difference))

Bloch norms coincide: True
Total difference: 1.2212453270876722e-15


In [7]:
R_r = compute_bloch_norms_from_vector(r_rec)

coincide = []
difference = []

for subset in subset_index_map.keys():
    difference.append(float(np.linalg.norm(R[subset] - R_r[subset])))
    coincide.append(np.allclose(R[subset], R_r[subset]))

print("Bloch norms coincide:", all(coincide))
print("Total difference:", sum(difference))

Bloch norms coincide: True
Total difference: 1.2212453270876722e-15
